# Part 1 — DEM: San Francisco Elevation Contours → H3

**A complete DEM download → elevation isobands → product-native H3 indexing → cumulative coverage tiers → per-tier rasterize → multi-band stack pipeline,
demonstrated on USGS 3DEP 10 m elevation data for San Francisco.**

## What this notebook shows

San Francisco is ideal terrain for elevation isobands: sea level at the Bay and Pacific coast rises
sharply through the Mission and Castro neighborhoods up to Twin Peaks (~280 m) and Mt Davidson
(~282 m). The ~300 m spread over the city AOI produces twelve meaningful 25 m elevation bands.

This pipeline produces **two complementary views**:

1. **Interval elevation-band map** — which single elevation band (e.g. 75–100 m) each H3 cell falls in.
   Useful for inspecting band geometry and cell coverage.
2. **Cumulative-tier depth map** — how many elevation thresholds each location clears.
   Tier T = "elevation ≥ BREAKS[T]"; a cell in band *i* belongs to tiers 0…*i*, so depth climbs
   with terrain height — the **telco nested-coverage model** applied to elevation.

**Pipeline steps:**

1. **Download a DEM** — `DemDownloader().download(SF_BBOX, DEM_DIR, resolution="finest")` fetches the
   USGS **3DEP seamless 10 m** dataset windowed to the San Francisco AOI via the Planetary Computer STAC
   API, staged to a UC Volume (idempotent). `FORCE_RELOAD=False` skips the download on re-runs.
2. **Read as a Spark DataFrame** — `.read(DEM_DIR)` returns a `tile` DataFrame (EPSG:4326, one row per tile).
3. **Extract elevation isobands** — `rx.rst_isoband("tile", breaks)` with 25 m breaks (0, 25, …, 300)
   produces **12 contiguous band polygons** as a distributed Spark column of WKB geometry.
4. **H3 indexing via Databricks product H3** — [`h3_try_coverash3`](https://docs.databricks.com/aws/en/sql/language-manual/functions/h3_try_coverash3)
   (overlap) fills each band polygon at **H3 resolution 10** (~65 m edge), producing solid, gap-free H3 cells per band.
   Fully distributed — no driver-side h3-py.
5. **Cumulative coverage tiers** — from `(band_level, cellid)` pairs, derive tiers where tier T = union
   of all cells with band_level ≥ T. A cell in band *i* belongs to tiers 0…*i*, so coverage depth
   climbs with elevation — the telco nested-coverage model.
6. **Shared grid spec** — `rx.rst_h3_gridspec` snaps a single pixel-aligned canvas over all bands combined.
7. **Per-tier rasterize** — `rx.rst_h3_rasterize_agg` burns each cumulative tier's H3 cells onto the
   shared canvas. `FORCE_RELOAD=False` reuses a materialized temp table for fast re-runs.
8. **Stack** — `rx.rst_frombands_agg` assembles per-tier tiles into a single multi-band GeoTIFF;
   `plot_raster(composite="depth")` now shows a real 1..N gradient — the number of elevation thresholds
   each pixel clears (coastal = 1, Twin Peaks = 12).

## Part 1 of 3

| Notebook | Input | Key GeoBrix function |
|---|---|---|
| **Part 1 — DEM** (this notebook) | USGS 3DEP 10 m raster | `rst_isoband` → product H3 |
| **Part 2 — LiDAR→DSM** | LiDAR point cloud | `rst_binpoints` |
| **Part 3 — CHM** | DSM − DTM | `rst_chm` |

> **Resolution note:** H3 res 10 (~65 m edge) suits terrain-scale visualisation. Telco and precision
> use cases at res 13–15 (~10 m–1 m) require the finer elevation detail produced in Parts 2 and 3.

## Runtime requirement

Run on **Databricks Serverless environment 5** (or any cluster with UC Volumes access). This notebook
uses the [lightweight execution tier](https://databrickslabs.github.io/geobrix/docs/api/execution-tiers) — no
JAR or GDAL init script required. (Serverless environment 6 works too once a known ipykernel restart
bug is resolved.) See the [GeoBrix API overview](https://databrickslabs.github.io/geobrix/docs/api/overview)
for the full function reference. Outputs are empty until executed on a cluster.

---
_Last Modified: September 17, 2026_

![H3 Rasterize — DEM isobands to a multi-band H3 raster stack](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/h3-rasterize/h3-rasterize.png)

## Install GeoBrix

The lightweight `[light_env5,vizx]` extras are sufficient — no JAR or GDAL init script
required (when using the lightweight tier). The notebook targets **Serverless environment 5**.
(Serverless environment 6 works too once a known ipykernel restart bug is resolved.)

In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix[light_env5,stac,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet "geobrix[light_env5,stac,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"

In [ ]:
%restart_python

## Imports and registration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import functions as F

# GeoBrix lightweight tier — runs on Serverless.
from databricks.labs.gbx.pyrx import functions as rx
from databricks.labs.gbx.pygx import functions as gx   # grid functions (h3_cellfill, ...)
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.vizx import plot_file, plot_raster, plot_static, plot_mask_layers, cells_as_gdf, grid_as_gdf

# Register pyrx SQL UDFs and H3 UDTFs (rst_h3_rastertogridavg, etc.).
rx.register(spark)
gx.register(spark)   # registers gbx_h3_cellfill (used in the fill-missing-cells option below)
register(spark)

In [ ]:
INTERACTIVE_PLOTS = False  # set True for interactive MapLibre maps

## DEM path and parameters

[`DemDownloader`](https://databrickslabs.github.io/geobrix/docs/sample-data/dem-downloader) wraps the **USGS 3DEP seamless** dataset hosted on Microsoft Planetary Computer.
With `resolution="finest"` it queries the 10 m product for tiles intersecting `SF_BBOX`, downloads
them, and writes GeoTIFF files to `DEM_DIR` in the UC Volume — idempotent on repeat runs.

> **Swap to 2 m LiDAR in one line:** `DemDownloader.lidar_dtm().download(SF_BBOX, DEM_DIR)` fetches the
> 3DEP-LiDAR-DTM product (~2 m resolution) to the same Volume layout, no other pipeline change needed.
> That 2 m DEM is the input for Part 2.

In [ ]:
# --- area & output ---
SF_BBOX = (-122.52, 37.70, -122.35, 37.83)          # San Francisco city (lon/lat)
DEM_DIR = "/Volumes/geospatial_docs/geobrix/sample-data/geobrix-examples/sf/elevation-3dep"

# --- parameters ---
H3_RESOLUTION = 10                                   # ~65 m edge (Telco wants 13-15; see Notebooks 2/3)
BAND_STEP_M   = 25
BREAKS        = [float(b) for b in range(0, 301, BAND_STEP_M)]   # 0,25,...,300 -> 12 bands
FORCE_RELOAD  = False  # True → re-download the DEM and rebuild materialized tables; False → reuse staged/cached artifacts (fast re-runs). Validation runs may pass --set-var FORCE_RELOAD=True.

# shared H3 canvas pixel size: ~3 px per res-10 hex edge (~65.9 m) at SF latitude
import math
_EDGE_M = 65.9
PIXEL_SIZE_DEG = (_EDGE_M / 3.0) / (111320.0 * math.cos(math.radians(37.8)))
print(f"H3 res {H3_RESOLUTION} | breaks {BREAKS} | pixel {PIXEL_SIZE_DEG:.7f} deg")

### Stage the 3DEP DEM

`DemDownloader().download(SF_BBOX, DEM_DIR, resolution="finest")` queries the Planetary Computer
STAC for USGS 3DEP seamless tiles intersecting `SF_BBOX`, downloads them, and writes the result to
the Volume — **once, idempotent** (skips files that already exist). Safe on Serverless: uses
rasterio's bundled GDAL, no CLI tools required. See the [DEM downloader reference](https://databrickslabs.github.io/geobrix/docs/sample-data/dem-downloader) for all download options and supported datasets.

In [ ]:
from databricks.labs.gbx.sample.dem import DemDownloader
import glob
# For finer bare-earth (~2 m, LiDAR-derived) swap the downloader — no other change:
#   dem = DemDownloader.lidar_dtm()   # 3dep-lidar-dtm; 2 m over SF (1 m where surveyed)
# Part 1 uses the 10 m seamless product (matches H3 res 10); 2 m feeds Part 2.
dem = DemDownloader()                                # 3dep-seamless (10/30 m)
existing = glob.glob(f"{DEM_DIR}/**/*.tif", recursive=True)
if FORCE_RELOAD or not existing:
    meta = dem.download(SF_BBOX, DEM_DIR, resolution="finest")   # live: Planetary Computer STAC
    meta.select("item_id", "out_file_path", "out_file_sz", "is_out_file_valid").show(truncate=False)
else:
    print(f"DEM already staged ({len(existing)} tile(s)) under {DEM_DIR} — skipping download (FORCE_RELOAD=False).")

## Step 1 — Load DEM and extract elevation isobands

`DemDownloader().read(DEM_DIR)` returns a **Spark DataFrame** of `tile` structs (EPSG:4326, one row
per downloaded tile). `rx.rst_isoband("tile", breaks)` (see [RasterX functions](https://databrickslabs.github.io/geobrix/docs/api/raster-functions)) runs as a **distributed Spark column
expression** — each executor processes one tile and emits rows of
`(band_level, geom_wkb, elev_lo, elev_hi)` for every contiguous elevation band.

With 25 m breaks from 0 to 300 m, the SF DEM yields **12 elevation bands**: from sea-level coastal
areas through the city's mid-slope neighborhoods up to the hilltops near Twin Peaks. Each row's
`geom_wkb` column holds the polygon as **WKB BINARY** — the native input format for Databricks
product H3 functions in the next step.

In [ ]:
tiles = DemDownloader().read(DEM_DIR)                # 'tile' struct, EPSG:4326 (1..N 3DEP tiles)
breaks_arr = F.array(*[F.lit(b) for b in BREAKS])
patches = (tiles
    .select(F.explode(rx.rst_isoband("tile", breaks_arr)).alias("p"))
    .select(F.col("p.band").alias("band_level"),
            F.col("p.geom_wkb").alias("geom_wkb"),
            F.col("p.lower").alias("elev_lo"),
            F.col("p.upper").alias("elev_hi")))
patches.groupBy("band_level", "elev_lo", "elev_hi").count().orderBy("band_level").show()

### Render the DEM

The staged DEM GeoTIFF rendered via [`plot_file`](https://databrickslabs.github.io/geobrix/docs/api/vizx) (auto-decimation + percentile stretch;
single-band → viridis). `DemDownloader.read()` returns **virtual tiles** — the `raster`
bytes are not materialized in the struct — so the staged `.tif` file is plotted directly.

In [ ]:
# DemDownloader.read() returns VIRTUAL tiles (raster bytes are not materialized in the
# struct), so render the staged GeoTIFF directly with plot_file (auto-decimation +
# percentile stretch). DEM_DIR is a UC Volume path; glob finds the downloaded tile(s).
import glob
dem_tifs = sorted(glob.glob(f"{DEM_DIR}/**/*.tif", recursive=True))
plot_file(dem_tifs[0], fig_w=8, fig_h=6)

**GeoBrix [`rst_isoband`](https://databrickslabs.github.io/geobrix/docs/api/raster-functions)** emits `geom_wkb` as **WKB BINARY** — the exact format Databricks product H3 functions
accept. The hand-off from GeoBrix geometry to Databricks H3 indexing uses the **product Python
bindings** (`pyspark.databricks.sql.functions`):

```python
from pyspark.databricks.sql import functions as DBF
h3_index = DBF.h3_try_coverash3   # every H3 cell that overlaps the polygon — solid, gap-free bands
```

**Product's [`h3_try_coverash3`](https://docs.databricks.com/aws/en/sql/language-manual/functions/h3_try_coverash3)** (a Databricks built-in) returns every H3 cell that **overlaps** the polygon, ensuring
solid, gap-free bands across the SF terrain. The `h3_try_*` prefix returns NULL for degenerate
patches instead of failing the job.

> **WKB, not GEOMETRY.** The product H3 functions accept WKB BINARY and **reject the GEOMETRY type** —
> do not wrap `geom_wkb` in `ST_GeomFromWKB`.

Both **GeoBrix `rst_isoband`** (Step 1) and the **product H3 indexing** run as **distributed Spark
expressions** — no driver-side `h3.polygon_to_cells`, no `collect()`. This is the **GeoBrix →
Databricks-native H3 on-ramp**: GeoBrix extracts the geometry; the Databricks platform handles
spatial indexing at scale. Available on **DBR 16.3+ / Serverless** (tested on Serverless env 5).

In [ ]:
# Hand off to Databricks-native H3 via the product Python bindings. rst_isoband emits
# WKB, which the H3 functions accept directly (no ST_GeomFromWKB). The h3_try_* variants
# return NULL instead of erroring on a bad/degenerate patch, so one sliver can't fail the
# whole job. Fully distributed — no driver-side H3 indexing.
from pyspark.databricks.sql import functions as DBF

h3_index = DBF.h3_try_coverash3
cells_df = (
    patches
    .select(
        "band_level",
        F.explode(h3_index(F.col("geom_wkb"), F.lit(H3_RESOLUTION))).alias("cellid"),
    )
    .where(F.col("cellid").isNotNull())
    .distinct()
)
cells_df.groupBy("band_level").count().orderBy("band_level").show()

### Cumulative coverage tiers: interval bands → nested depth model

GeoBrix `rst_isoband` produces **disjoint** elevation intervals — each pixel belongs to exactly one band —
so stacking the 12 per-band rasters gives a coverage depth of ≈ 1 everywhere (one flat colour,
no gradient). To get a meaningful depth signal we convert to **cumulative tiers**:

- **Tier T** = "elevation ≥ `BREAKS[T]`" = union of all interval bands ≥ T
- A cell in interval band *i* therefore belongs to tiers *0 … i*
- **Coverage depth** (# tiers covering a pixel) climbs with elevation — low coastal terrain hits
  one threshold; the Twin Peaks hilltops clear all twelve

This mirrors the **telco nested-coverage model** — a site covered by three operators is counted once
per operator, and the depth map shows which areas have the most options. Here, "operators" are
elevation thresholds and "coverage" is terrain height.

In [ ]:
# Cumulative coverage tiers: tier T = "elevation >= breaks[T]" = union of interval bands >= T.
# A cell in interval band i belongs to tiers 0..i, so coverage DEPTH (# tiers covering a
# pixel) climbs with elevation — the telco nested-coverage model.
tiers_df = (
    cells_df
    .select("cellid", F.explode(F.sequence(F.lit(0), F.col("band_level"))).alias("tier"))
    .distinct()
)
tiers_df.groupBy("tier").count().orderBy("tier").show()

### Render the H3 cells

`plot_static` draws the H3 cells straight from the Spark DataFrame: with
`grid_system="h3"` it turns each `cellid` into its hexagon boundary and colours it by
elevation band, over a basemap (also can plot geoms). With the default `coveras` mode the
bands form solid, contiguous fills — no white gaps between hexagons. Each cell keeps its own geometry, so this per-cell view
is handy for inspecting individual hexagons at this data size. It renders as a static
image (so it shows on GitHub and the docs site); for an interactive pan/zoom map, pass the
same Spark DataFrame to `vizx.plot_interactive(cells_df, grid_system="h3", column="band_level")`
— it handles scale (falling back to a raster overlay for very large cell sets) and renders
inline in Databricks.

__Note:__ for much larger cell sets, dissolve first —
`cells_as_gdf(cells_df, "cellid", extra_cols=["band_level"], dissolve_by="band_level", max_rows=200_000)`
merges each band into a single footprint polygon (far fewer geometries to render), then
pass that GeoDataFrame to `plot_static`. (`plot_static` itself renders one hexagon per
cell; dissolving lives on the `cells_as_gdf` adapter.) Cell maps in this notebook pass
`max_rows=200_000` explicitly — VizX's default caps at 10 000 and would silently drop cells
before the dissolve at SF scale (~13–18 k cells), leaving visible gaps in the output.

> **Gaps here can be filled** with GeoBrix `h3_cellfill` — demonstrated in the **Cumulative Tier
> Overlay** section below, on the per-cell coverage-**depth** surface (where each `cellid` is unique,
> so the fill has nothing to collapse). It is deliberately *not* applied to this interval-band
> surface: with `coveras` a cell can belong to several adjacent bands, so a `cellid` repeats and a
> value-per-cell fill would collapse those duplicates.

In [ ]:
# Dissolve per-cell hexagons into one footprint polygon per band before rendering.
# cells_as_gdf with dissolve_by="band_level" merges each band into one polygon, so
# plot_static receives at most 12 rows instead of ~13–18k hex boundaries.
# max_rows=200_000: the SF run produces ~13–18k cells; the VizX default of 10k would
# silently truncate before dissolving, leaving chunks of missing data in the render.
cells_gdf = cells_as_gdf(cells_df, "cellid", extra_cols=["band_level"], dissolve_by="band_level", max_rows=200_000)
ax = plot_static(
    cells_gdf,
    column="band_level",
    cmap="viridis",
    title="H3 cells by elevation band",
)

## Step 3 — Shared grid spec via `rst_h3_gridspec`

[`rst_h3_gridspec`](https://databrickslabs.github.io/geobrix/docs/api/raster-functions) computes a pixel-snapped bounding box and pixel dimensions
that span **all** H3 cells across **all** band levels. Because we pass no
grouping column here (we want a single shared canvas), it returns a single-row
DataFrame with a `grid` struct.

Using the same `grid` for every band ensures all per-band tiles are spatially
aligned — a prerequisite for `rst_frombands_agg` to produce a coherent stack.

In [ ]:
# Compute the shared canvas over ALL cells (no grouping column → one grid row).
grid_df = rx.rst_h3_gridspec(cells_df, "cellid", pixel_size=PIXEL_SIZE_DEG)
grid_row = grid_df.first()
g = grid_row["grid"]

print("Shared canvas grid spec:")
print(f"  xmin={g['xmin']:.6f}  ymin={g['ymin']:.6f}")
print(f"  xmax={g['xmax']:.6f}  ymax={g['ymax']:.6f}")
print(f"  pixel_size={g['pixel_size']:.6f}")
print(f"  width={g['width']}  height={g['height']}  srid={g['srid']}")

# Broadcast the grid constants to all rows so rst_h3_rasterize_agg
# receives the same extent per group.
cells_with_grid = cells_df.withColumn("xmin",   F.lit(g["xmin"]))\
                           .withColumn("ymin",   F.lit(g["ymin"]))\
                           .withColumn("xmax",   F.lit(g["xmax"]))\
                           .withColumn("ymax",   F.lit(g["ymax"]))\
                           .withColumn("width",  F.lit(g["width"]))\
                           .withColumn("height", F.lit(g["height"]))

# Same broadcast to tiers_df for per-tier rasterize.
tiers_with_grid = tiers_df.withColumn("xmin",   F.lit(g["xmin"]))\
                           .withColumn("ymin",   F.lit(g["ymin"]))\
                           .withColumn("xmax",   F.lit(g["xmax"]))\
                           .withColumn("ymax",   F.lit(g["ymax"]))\
                           .withColumn("width",  F.lit(g["width"]))\
                           .withColumn("height", F.lit(g["height"]))

### Render the shared canvas over the cells
  
`rst_h3_gridspec` returns the shared raster canvas — the bounding rectangle (and
pixel grid) every per-band raster aligns to. Drawing that rectangle over the cells
shows the extent the H3 cells get binned into: we render the cells with
`plot_static`, then overlay the canvas outline on the same axes.

`plot_static` reprojects layers to Web Mercator (EPSG:3857) so they line up with
the basemap, so the boundary overlay is reprojected to match before drawing.

In [ ]:
# Multi-layer (static): per-band H3 footprints + the shared-canvas rectangle on
# one matplotlib axes, over a basemap. Static so it renders on GitHub + the docs (see
# the note above); for an interactive pan/zoom map, build GeoDataFrames with
# cells_as_gdf(...) / grid_as_gdf(...) and pass to plot_interactive([vector_layer(...)]).
grid_gdf = grid_as_gdf(g)  # g = grid_row["grid"] -> the shared canvas spec

# Dissolve into one footprint polygon per band; max_rows=200_000 ensures the full SF
# cell set (~13–18k) is included before the dissolve (VizX default=10k would truncate).
cells_gdf = cells_as_gdf(cells_df, "cellid", extra_cols=["band_level"], dissolve_by="band_level", max_rows=200_000)

# Cells coloured by elevation band over a basemap, then the shared canvas as a red
# OUTLINE (fill=False so it doesn't cover the cells; basemap=False so it doesn't
# re-fetch tiles). Both layers reproject to 3857 and align.
ax = plot_static(
    cells_gdf,
    column="band_level",
    cmap="viridis",
    title="H3 cells by elevation band + shared canvas",
)
ax = plot_static(grid_gdf, ax=ax, fill=False, edgecolor="red", basemap=False)

## Step 4 — Per-tier rasterize with `rst_h3_rasterize_agg`

[`rst_h3_rasterize_agg`](https://databrickslabs.github.io/geobrix/docs/api/raster-functions) burns a group's H3 cells onto the shared canvas,
producing a presence mask (1.0 where a cell's centroid falls, NoData elsewhere).
Supplying the explicit extent from `rst_h3_gridspec` guarantees every per-tier
tile has identical pixel dimensions — which is required for stacking.

We group by `tier` so each of the **12 cumulative elevation thresholds** produces one
pixel-aligned tile: tier 0 = "elevation ≥ 0 m" (covers the full AOI), tier 11 = "elevation ≥ 275 m"
(only the Twin Peaks / Mt Davidson hilltops).

Because the per-pixel rasterize is expensive and we read these tiles again in the
next two steps, we materialize them once into a **session-scoped [temporary table](https://docs.databricks.com/aws/en/tables/temporary-tables)**
(`CREATE TEMP TABLE`). On Serverless, `.cache()` / `.persist()` are unavailable, so a
temp table is the idiomatic way to avoid recomputing the burn; it is automatically
dropped when the session ends. Requires Serverless or DBR 18.1+ (not dedicated /
single-user clusters).

When `FORCE_RELOAD=False` and the table already exists in the session, the
materialization step is skipped — ideal for iterating on visualization cells without
re-running the expensive burn.

In [ ]:
# One rasterized tile per cumulative tier, all on the shared canvas.
# FORCE_RELOAD guard: re-use the materialized tier_tiles if already computed this session.
if FORCE_RELOAD or not spark.catalog.tableExists("tier_tiles"):
    _tiers_tiles = tiers_with_grid.groupBy("tier").agg(
        rx.rst_h3_rasterize_agg(
            "cellid",
            xmin="xmin",
            ymin="ymin",
            xmax="xmax",
            ymax="ymax",
            width="width",
            height="height",
        ).alias("tile")
    )
    _tiers_tiles.createOrReplaceTempView("_tier_src")
    spark.sql("CREATE OR REPLACE TEMP TABLE tier_tiles AS SELECT * FROM _tier_src")
else:
    print("reusing tier_tiles (FORCE_RELOAD=False)")
tier_tiles = spark.table("tier_tiles").orderBy("tier")

print(f"Tier tiles: {tier_tiles.count()} rows")
tier_tiles.select("tier",
                  F.expr("gbx_rst_width(tile)    AS px_width"),
                  F.expr("gbx_rst_height(tile)   AS px_height"),
                  F.expr("gbx_rst_numbands(tile) AS numbands")).show()

### Cumulative tier overlay

We render **all cumulative tiers** in a single nested overlay — tier 0 (full AOI, lowest
threshold) drawn first, tier 11 (highest hilltops only) on top — using a sequential `viridis`
colormap. Each tier is labelled `>= X m`, where X is the elevation threshold.

The result reads like a **telco signal-strength heatmap**: the outermost (lowest-elevation) tier
covers the entire AOI and the concentric inner rings shrink progressively toward the hilltops at
Twin Peaks and Mt Davidson. Darker hues mark areas at or above sea level; the brightest inner
ring marks terrain above 275 m.

In [ ]:
# All cumulative tiers overlaid, tier 0 (largest) first → tier 11 (smallest hilltops) on top.
# Sequential viridis colormap: outer (low) tiers get dark hues, inner (high) tiers get bright.
import matplotlib

tiers_sorted = sorted(r["tier"] for r in tier_tiles.select("tier").distinct().collect())
cmap = matplotlib.colormaps["viridis"]
n = max(len(tiers_sorted) - 1, 1)
layers, colors = [], []
for k, t in enumerate(tiers_sorted):
    row = tier_tiles.filter(F.col("tier") == t).first()
    layers.append((f">= {int(BREAKS[t])} m", row["tile"]["raster"]))
    colors.append(cmap(k / n))

plot_mask_layers(layers, colors=colors, fig_w=9, fig_h=7,
                 title="Cumulative coverage tiers (elevation >= threshold)")

### Option — fill missing cells with GeoBrix `h3_cellfill`

The cumulative tiers give us a **per-cell coverage depth**: how many tiers cover each cell — **one
value per `cellid`**. That uniqueness is exactly what a value-per-cell fill needs. (It is *not* true
of the interval `band_level` surface, where `coveras` places a cell in several adjacent bands, so a
`cellid` repeats — a value-per-cell fill would collapse those duplicates. Depth avoids that entirely.)

**GeoBrix `h3_cellfill`** (a lightweight-tier grouped aggregator) can *optionally* fill **missing
cells** — cells with no depth value — by interpolating each empty cell's value from its valid
**in-group neighbours within `k` rings**. We keep it strictly **local — `k=1` with inverse-distance
weighting (IDW)** — so a gap adopts only its *immediate* neighbours ("water fills with water"); a gap
with no filled neighbour within one ring stays empty, so the fill never invents coverage.

`h3_cellfill` fills NULL-valued cells rather than generating them, so we feed it the present cells
plus their 1-ring neighbours as `NULL`-valued gap candidates. Those neighbours come from **Product's
[`h3_kring`](https://docs.databricks.com/aws/en/sql/language-manual/functions/h3_kring)** (a Databricks
built-in). Shown purely as an **option**; the pipeline works without it.

In [ ]:
# ── OPTION: fill missing cells with GeoBrix h3_cellfill (k=1, IDW) ──────────────
# Per-cell coverage DEPTH = number of cumulative tiers covering the cell — ONE value
# per cellid (unlike the interval band_level, where coveras repeats a cellid across
# adjacent bands). cellid is unique here, so nothing collapses in the fill.
from pyspark.databricks.sql import functions as DBF
from databricks.labs.gbx.pygx import _cellfill

depth_df = tiers_df.groupBy("cellid").agg(F.count("*").cast("double").alias("depth"))

# Gap candidates = 1-ring neighbours (Product's h3_kring) not already present; feed them
# as (cellid, NULL). GeoBrix h3_cellfill fills each from its in-group neighbours within
# k=1 rings (IDW, power=2). k=1 keeps it strictly local ("water fills with water").
gap = (depth_df.select(F.explode(DBF.h3_kring(F.col("cellid"), F.lit(1))).alias("cellid"))
       .distinct().join(depth_df.select("cellid"), on="cellid", how="left_anti")
       .withColumn("depth", F.lit(None).cast("double")))
fill_in = depth_df.unionByName(gap).withColumn("g", F.lit(1))
payload = fill_in.groupBy("g").agg(
    gx.h3_cellfill("cellid", "depth", F.lit(1), F.lit("idw"), F.lit(2.0)).alias("p")
).first()["p"]
rows = [(int(cid), v) for cid, v in _cellfill.decode(payload) if v is not None]   # drop unfilled gaps
filled_depth = (spark.createDataFrame(rows, "cellid long, depth double")
                .withColumn("depth", F.round("depth").cast("int")))
print(f"present={depth_df.count()}  gap_candidates={gap.count()}  filled={filled_depth.count()}")

# Dissolve by the integer depth (<=~12 groups); max_rows=200_000 ensures the full
# filled cell set is included before the dissolve (VizX default=10k would truncate).
filled_gdf = cells_as_gdf(filled_depth, "cellid", extra_cols=["depth"], dissolve_by="depth", max_rows=200_000)
ax = plot_static(
    filled_gdf,
    column="depth",
    cmap="viridis",
    title="Coverage depth with missing cells filled — GeoBrix h3_cellfill (k=1, IDW)",
)

## Step 5 — Stack tiers with `rst_frombands_agg`

[`rst_frombands_agg`](https://databrickslabs.github.io/geobrix/docs/api/raster-functions) assembles the per-tier tiles into a single multi-band
GeoTIFF, ordered by `tier` ascending. Band *i* in the stacked raster corresponds to
cumulative tier *i* ("elevation ≥ `BREAKS[i]`").

Unlike stacking **interval** bands (which would give depth ≈ 1 everywhere — each pixel covered
by exactly one disjoint band), stacking **cumulative** tiers gives a real gradient: each pixel's
coverage depth equals the **number of elevation thresholds it clears** — low coastal terrain has
depth 1 (only the ≥ 0 m tier covers it) while the Twin Peaks hilltops reach depth 12 (every
threshold cleared). `plot_raster(composite="depth")` renders this directly.

In [ ]:
# Add a sequential band_index starting at 1 so rst_frombands_agg orders correctly.
from pyspark.sql.window import Window

w = Window.orderBy("tier")
indexed = tier_tiles.withColumn(
    "band_index",
    F.row_number().over(w).cast("int"),
)

# Stack all per-tier tiles into one multi-band tile.
stacked_df = indexed.agg(
    rx.rst_frombands_agg("tile", "band_index").alias("stacked")
)

stacked_row = stacked_df.first()
stacked_tile = stacked_row["stacked"]
print("Stacked tile schema:", stacked_tile.asDict().keys())

# Quick metadata check via SQL accessors.
meta = stacked_df.selectExpr(
    "gbx_rst_numbands(stacked) AS n_bands",
    "gbx_rst_width(stacked)    AS width",
    "gbx_rst_height(stacked)   AS height",
    "gbx_rst_srid(stacked)     AS srid",
).first()
print(f"Bands: {meta['n_bands']}  Width: {meta['width']}  Height: {meta['height']}  SRID: {meta['srid']}")

### Visualize the stacked multi-band raster — coverage depth

`plot_raster` with `composite="depth"` renders **coverage depth**: the per-pixel count of
cumulative tiers that cover each pixel. Because each tier T = "elevation ≥ BREAKS[T]",
depth is the **number of elevation thresholds the terrain clears**:

- **Depth 1** — coastal and sea-level areas: only the ≥ 0 m tier covers them
- **Depth ~6** — mid-slope neighborhoods (~125 m): six thresholds cleared
- **Depth 12** — Twin Peaks and Mt Davidson hilltops: all twelve thresholds cleared

The result is a viridis gradient rising with elevation — a direct visual proxy for terrain
height. Uncovered pixels (ocean and areas outside the SF AOI) are masked transparent.

This is the **telco nested-coverage model** applied to elevation: depth = how many coverage
tiers overlap at a location. In a real telco deployment each tier would be a different
operator or signal-strength class; here the tiers are elevation thresholds and depth is a
continuous proxy for terrain height.

In [ ]:
# Render the stacked raster as a coverage-depth map.
# composite="depth" counts, per pixel, how many cumulative tiers cover it → viridis gradient.
# Uncovered pixels (NoData in all tiers) are masked transparent.
plot_raster(stacked_tile["raster"], composite="depth", fig_w=10, fig_h=8)

Multi-layer view: the coverage-depth raster overlaid with H3 cell boundaries by elevation band — pan and zoom to explore the spatial distribution.

In [ ]:
if INTERACTIVE_PLOTS:
    from databricks.labs.gbx.vizx import raster_layer, grid_layer, plot_interactive
    plot_interactive([
        raster_layer(stacked_tile["raster"]),
        grid_layer(cells_df, grid_system="h3", cellid_col="cellid", column="band_level", opacity=0.25),
    ])

## Summary

This notebook demonstrated the full H3-cell rasterization pipeline on a San Francisco DEM,
producing **12 elevation bands** (0–300 m, 25 m step) at H3 resolution 10 using USGS 3DEP
seamless 10 m elevation data, with two complementary views:

> **Function provenance.** `rst_*`, `rst_h3_*`, and `h3_cellfill` are **GeoBrix** functions;
> `h3_try_coverash3` and `h3_kring` are **Databricks product** (built-in) H3 functions, called via
> the `DBF` (`pyspark.databricks.sql.functions`) binding.

**View 1 — Interval elevation-band map** (which single band each area falls in):

| Step | Function | Output |
|---|---|---|
| Download DEM (staged once) | `DemDownloader().download(…)` | GeoTIFF tiles in UC Volume |
| DEM → isobands (Spark) | GeoBrix `rx.rst_isoband` | `(band_level, geom_wkb)` rows |
| Polygon → H3 cells (Spark) | Product [`h3_try_coverash3`](https://docs.databricks.com/aws/en/sql/language-manual/functions/h3_try_coverash3) | `(band_level, cellid)` rows |
| Dissolve + render (viz) | `cells_as_gdf` + `plot_static` | H3 cells coloured by band |

**View 2 — Cumulative-tier depth map** (how many elevation thresholds each location clears):

| Step | Function | Output |
|---|---|---|
| Interval → cumulative tiers | `F.explode(F.sequence(0, band_level))` | `(tier, cellid)` rows |
| Shared canvas (Spark) | GeoBrix `rx.rst_h3_gridspec` | `grid` struct (xmin…height) |
| H3 cells → tile (Spark) | GeoBrix `rx.rst_h3_rasterize_agg` | single-band tile per tier |
| Stack tiers (Spark) | GeoBrix `rx.rst_frombands_agg` | multi-band GeoTIFF tile |
| Nested overlay (viz) | `plot_mask_layers` | concentric tier footprints + legend |
| Fill missing cells (viz option) | GeoBrix `h3_cellfill` + Product `h3_kring` | depth surface with gaps filled |
| Depth map (viz) | `plot_raster(composite="depth")` | 1..12 coverage-depth gradient |

The depth gradient is meaningful because each cumulative tier T = "elevation ≥ BREAKS[T]" — a pixel's
depth count equals the number of elevation thresholds it clears. Low coastal terrain has depth 1;
Twin Peaks / Mt Davidson reach depth 12. This is the **telco nested-coverage model** applied to
terrain: depth = how many coverage tiers overlap at a location.

## What's next — Parts 2 and 3

This notebook uses **USGS 3DEP seamless 10 m** elevation. The 65 m H3 hexagons at res 10 suit
terrain-scale visualisation but are too coarse for precision use cases (res 13–15, ~10 m–1 m),
which require the finer elevation detail available in the rest of the series:

| Notebook | Input | New function | Why |
|---|---|---|---|
| **Part 2 — LiDAR→DSM** | LiDAR point cloud (~1–2 m) | `rst_binpoints` | Bin millions of LiDAR returns into a DSM raster |
| **Part 3 — CHM** | DSM − DTM from Parts 1/2 | `rst_chm` | Canopy height model for urban vegetation mapping |

> H3 res 13–15 for Telco planning pairs naturally with the 1–2 m LiDAR elevation of Parts 2 and 3.
> See the [LiDAR reader docs](https://databrickslabs.github.io/geobrix/docs/readers/lidar) for the
> Part 2 point-cloud input pipeline.

### Further reading

- [GeoBrix RasterX functions](https://databrickslabs.github.io/geobrix/docs/api/raster-functions) — full function reference
- [EO-Series notebooks](https://databrickslabs.github.io/geobrix/docs/notebooks/eo-series) — STAC download, band stacking, clipping pipeline